# Lead Time Check: `yearlyFullLoadHoursMin`

This notebook is based on `12_leadTimes_example.ipynb` and keeps the setup as close as possible to the original example.

The configuration is the following:
- Two locations: `location1` and `location2`
- One commodity: `commodity1` with unit `unit1`
- Two investment periods: `0` and `1`, each representing one year
- One source `source1` producing `commodity1`
- One sink `sink1` consuming `commodity1`
- `source1` has `leadTime = 1`
- `source1` has `commissioningFix = 1` in IP 0 and `commissioningFix = 0` in IP 1, so capacity should become available in IP 1
- `source1` has `yearlyFullLoadHoursMin = 8760`
- `sink1` has no fixed demand; it only provides an optional outlet in IP 1

What is checked:
- Whether `yearlyFullLoadHoursMin` is applied to physically available `capacity` rather than directly to `commis`
- Whether the full-load-hour constraint is inactive in IP 0 because available capacity should still be 0
- Whether the full-load-hour constraint forces source operation in IP 1 once capacity is available

Expected behaviour with the current lead-time implementation for normal, non-commissioning-dependent operation:
- `commissioning` of `source1` should be 1 in IP 0 and 0 in IP 1
- `capacity` of `source1` should be 0 in IP 0 and 1 in IP 1
- `operation_annual` of `source1` should be 0 in IP 0
- `operation_annual` of `source1` should be 8760 in IP 1 for each location
- The ratio `operation_annual / capacity` should therefore be 8760 in IP 1

Interpretation:
- If the result follows this pattern, `yearlyFullLoadHoursMin` for normal operation is coupled to available capacity and is therefore less problematic than `commissioningFix` / `commissioningMin`.
- This notebook does not test `commissioningDependentCcf=True`; that case may behave differently because some constraints then use commissioning-dependent operation variables.


In [1]:
%load_ext autoreload
%autoreload 2

import fine as fn  # Provides objects and functions to model an energy system
import pandas as pd  # Used to manage data in tables
import numpy as np


In [2]:
esM = fn.EnergySystemModel(
    locations = {"location1", "location2"},
    commodities = {"commodity1"},
    commodityUnitsDict = {"commodity1": "unit1"},
    startYear = 0,
    numberOfInvestmentPeriods = 2,
    investmentPeriodInterval = 1
)


In [3]:
esM.add(
    fn.Source(
        esM = esM,
        name = "source1",
        commodity = "commodity1",
        hasCapacityVariable = True,
        operationRateMax = {0: pd.DataFrame(np.ones((8760, 2)), columns=["location1", "location2"], index=range(8760)),
                            1: pd.DataFrame(np.ones((8760, 2)), columns=["location1", "location2"], index=range(8760))
                            },
        capacityMax = {0: pd.Series({"location1": 100,
                                     "location2": 100}),
                       1: pd.Series({"location1": 100,
                                     "location2": 100})
                       },
        # Capacity is constructed in IP 0 and becomes available in IP 1 because leadTime = 1.
        commissioningFix = {0: pd.Series({"location1": 1,
                                          "location2": 1}),
                            1: pd.Series({"location1": 0,
                                          "location2": 0})
                            },
        # Test parameter: full-load-hour minimum.
        # With capacity = 1 in IP 1 and operationRateMax = 1, this should force full operation in IP 1.
        yearlyFullLoadHoursMin = 8760,
        investPerCapacity = 1000,
        opexPerCapacity = 1020,
        interestRate = 0.08,
        economicLifetime = 1,
        leadTime = {0: pd.Series({"location1": 1, "location2": 1}),
                    1: pd.Series({"location1": 1, "location2": 1})
                    }
    )
)


In [4]:
esM.add(
    fn.Sink(
        esM = esM,
        name = "sink1",
        commodity = "commodity1",
        hasCapacityVariable = False,
        # No fixed demand: the sink only provides a possible outlet for source operation.
        # Therefore, source operation in IP 1 should be caused by yearlyFullLoadHoursMin, not by demand.
        operationRateMax = {0: pd.DataFrame(np.zeros((8760, 2)), columns=["location1", "location2"], index=range(8760)),
                            1: pd.DataFrame(np.ones((8760, 2)), columns=["location1", "location2"], index=range(8760))
                            },
    )
)


In [5]:
solver = fn.utils.ImplementedSolvers.STANDARD_SOLVER.value

esM.optimize(timeSeriesAggregation=False, solver='gurobi')


Set parameter Threads to value 3
Set parameter LogFile to value ""
Set parameter QCPDual to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 3 threads

Non-default parameters:
QCPDual  1
Threads  3

Optimize a model with 70096 rows, 70096 columns and 175228 nonzeros (Min)
Model fingerprint: 0x87c2e378
Model has 4 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 9e+03]
  Objective range  [2e+03, 2e+03]
  Bounds range     [1e+00, 1e+02]
  RHS range        [0e+00, 0e+00]

Presolve removed 70096 rows and 70096 columns
Presolve time: 0.04s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.2000000e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.07 seconds (0.04 work u

In [8]:
def show_relevant_summary(esM, ip):
    summary = esM.getOptimizationSummary("SourceSinkModel", outputLevel=1, ip=ip)
    relevant_rows = [
        "commissioning", "capacity", "operation", "operation_annual",
        "invest", "capexCap", "opexCap", "capexIfBuilt", "opexIfBuilt",
        "TAC", "NPVcontribution",
    ]
    mask = summary.index.get_level_values(1).isin(relevant_rows)
    display(summary.loc[mask])
    return summary

summary_ip0 = show_relevant_summary(esM, ip=0)
summary_ip1 = show_relevant_summary(esM, ip=1)


location1 location2
Component Property         Unit                            
sink1     NPVcontribution  [1e9 Euro]         0.0       0.0
          TAC              [1e9 Euro/a]       0.0       0.0
          operation        [unit1*h]          0.0       0.0
          operation_annual [unit1*h/a]        0.0       0.0
source1   NPVcontribution  [1e9 Euro]      2100.0    2100.0
          TAC              [1e9 Euro/a]    2100.0    2100.0
          capacity         [unit1]            0.0       0.0
          capexCap         [1e9 Euro/a]    1080.0    1080.0
          commissioning    [unit1]            1.0       1.0
          invest           [1e9 Euro]      1000.0    1000.0
          operation        [unit1*h]          0.0       0.0
          operation_annual [unit1*h/a]        0.0       0.0
          opexCap          [1e9 Euro/a]    1020.0    1020.0

location1 location2
Component Property         Unit                            
sink1     NPVcontribution  [1e9 Euro]         0.0       0.0
          TAC              [1e9 Euro/a]       0.0       0.0
          operation        [unit1*h]       8760.0    8760.0
          operation_annual [unit1*h/a]     8760.0    8760.0
source1   NPVcontribution  [1e9 Euro]         0.0       0.0
          TAC              [1e9 Euro/a]       0.0       0.0
          capacity         [unit1]            1.0       1.0
          capexCap         [1e9 Euro/a]       0.0       0.0
          commissioning    [unit1]            0.0       0.0
          invest           [1e9 Euro]         0.0       0.0
          operation        [unit1*h]       8760.0    8760.0
          operation_annual [unit1*h/a]     8760.0    8760.0
          opexCap          [1e9 Euro/a]       0.0       0.0

In [7]:
# Optional manual check: operation_annual / capacity should be 8760 in IP 1.
# Depending on the exact FINE output format, inspect the displayed summary above.
